# 01 - EDA, qualidade dos dados e integração

Objetivos deste notebook:

- Avaliar qualidade, volume e cobertura das tabelas.
- Entender período histórico, status dos pedidos, receitas, reviews e prazos.
- Construir uma base integrada no nível de pedido para as próximas etapas.

In [ ]:
from pathlib import Path
import sys
import warnings

import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from matplotlib.ticker import PercentFormatter

import pandas as pd
import seaborn as sns
from IPython.display import display
pd.set_option('display.max_columns', None)

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_PATH = PROJECT_ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

# FunÃ§Ãµes para auxiliar as anÃ¡lises
from olist.config import FIGURES_DIR, INTERIM_DATA_DIR, ensure_project_dirs
from olist.data import build_order_level_dataset, load_all
from olist.features import add_delivery_features, add_review_target, add_temporal_features
from olist.plots import save_current_figure, set_plot_style

ensure_project_dirs()
set_plot_style()

In [ ]:
# Avaliação de qualidade dos dados
tables = load_all(include_geolocation = False)

quality_rows = []
for name, df in tables.items():
    quality_rows.append({
        'tabela': name,
        'qtde_linhas': len(df),
        'qtde_colunas': df.shape[1],
        'linhas_duplicadas': int(df.duplicated().sum()),
        'celulas_vazias': int(df.isna().sum().sum()),
        'pct_vazias': round(df.isna().mean().mean() * 100, 2),
    })

quality_df = pd.DataFrame(quality_rows).sort_values('qtde_linhas', ascending = False)
display(quality_df)

## Pedidos: período, status e sazonalidade

Aqui verifico a janela temporal do dataset e a distribuição dos status de pedido. Isso me ajudará a decidir filtros e evitar conclusões sobre perÃ­odos incompletos.

In [ ]:
# Cria a features temporais
orders = add_temporal_features(tables['orders'])

# Calcula a quantidade de pedidos por status
status_summary = (
    orders['order_status']
    .value_counts(dropna = False)
    .rename_axis('order_status')
    .reset_index(name = 'orders')
)
status_summary['pct'] = (status_summary['orders'] / status_summary['orders'].sum()) * 100
display(status_summary)

monthly_orders = orders.groupby('purchase_month').size().reset_index(name = 'orders')
monthly_orders['purchase_month'] = monthly_orders['purchase_month'].astype(str)


ax = sns.lineplot(
    data=monthly_orders,
    x='purchase_month',
    y='orders',
    marker='o',
    linewidth=2.5,
    markersize=8,
    color='#2563eb'
)

ax.set_title(
    'Pedidos por mÃªs',
    fontsize=15,
    weight='bold'
)

ax.set_xlabel('')
ax.set_ylabel('Orders')
ax.tick_params(axis='x', rotation=45)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
save_current_figure(FIGURES_DIR / '01_pedidos_por_mes.png')
plt.show()

##### Observações

- É possível notar que a variável `order_status` está altamente desbalanceada, pois ~97% dos registros pertencem à categoria `delivered` (entregues). Os demais status representam, em grande parte, exceções ou ruídos operacionais.  
Portanto: a análise de experiência do cliente será focada em pedidos entregues. Os demais status serão removidos do modelo analítico principal.

- O gráfico de pedidos por mês indica uma queda abrupta nos dois últimos meses, saindo de aproximadamente ~6,5 mil pedidos para próximo de zero.  
Portanto: optou-se pela remoção desses dois meses, pois há forte indício de dados incompletos, o que poderia enviesar análises temporais e interpretações de tendência.

- Também é possível perceber, através da evolução temporal dos pedidos, que o negócio começou a expandir de forma mais acelerada no final de 2016, atingindo pico ao longo de 2017.  
Portanto: análises gerenciais e comparações temporais devem considerar o contexto de expansão operacional da plataforma.

### Conclusão

A série temporal mostra crescimento consistente do volume de pedidos, indicando expansão do negócio ao longo do tempo. No entanto, há uma queda abrupta nos meses finais da base, sugerindo incompletude dos dados — motivo pelo qual esses períodos serão removidos das análises para evitar viés.

Além disso, a base é altamente concentrada em pedidos entregues (~97%), direcionando a análise de experiência do cliente para esse subconjunto. Os demais status serão tratados como exceções operacionais e removidos das análises principais, dado o contexto e objetivo inicial do proje

## Integração no ní­vel de pedido

Base integrada que consolida informações de cliente, itens, pagamentos e reviews em uma linha por `order_id`.

In [ ]:
# Integra as tabelas principais no nÃ­vel de pedido
order_level = build_order_level_dataset(tables)

# Cria alvo binÃ¡rio para baixa satisfaÃ§Ã£o <=2
# Cria variaveis de prazo, atraso e velocidade de entrega
order_level = add_review_target(add_delivery_features(add_temporal_features(order_level)))

integration_checks = pd.DataFrame({
    'metrica': ['pedidos', 'id_unico_de_pedido', 'linhas_com_avaliacao', 'linhas_com_itens', 'linhas_com_pgto'],
    'valor': [
        len(order_level),
        order_level['order_id'].nunique(),
        order_level['review_score'].notna().sum(),
        order_level['products_value'].notna().sum(),
        order_level['payment_value'].notna().sum(),
    ],
})
display(integration_checks)

In [ ]:
# Analise somente dos pedidos entregues
delivered = order_level[order_level['order_status'].eq('delivered')].copy()

# Optei por remover as linhas que nÃ£o possuem avaliaÃ§Ã£o, pois esses registros nÃ£o terÃ£o utilidade no presente estudo
delivered = delivered[delivered['review_score'].notna()]
delivered = delivered[delivered['payment_value'].notna()]

In [ ]:
integration_checks = pd.DataFrame({
    'metrica': ['pedidos', 'id_unico_de_pedido', 'linhas_com_avaliacao', 'linhas_com_itens', 'linhas_com_pgto'],
    'valor': [
        len(delivered),
        delivered['order_id'].nunique(),
        delivered['review_score'].notna().sum(),
        delivered['products_value'].notna().sum(),
        delivered['payment_value'].notna().sum(),
    ],
})
display(integration_checks)

## Receita, frete e categorias

A receita aqui considera valor de produtos

In [ ]:
monthly_revenue = (
    delivered.groupby('purchase_month')
    .agg(orders=('order_id', 'nunique'), products_value=('products_value', 'sum'), freight_value=('freight_value', 'sum'))
    .reset_index()
)
monthly_revenue['purchase_month'] = monthly_revenue['purchase_month'].astype(str)

# Tema visual
sns.set_theme(style='whitegrid')

# Formatter para moeda
def currency_formatter(x, pos):
    return f'R$ {x:,.0f}'.replace(',', '.')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# =========================
# Gráfico 1 - Produtos
# =========================
sns.lineplot(
    data=monthly_revenue,
    x='purchase_month',
    y='products_value',
    marker='o',
    linewidth=2.5,
    markersize=8,
    color='#2563eb',
    ax=axes[0]
)

axes[0].set_title(
    'Valor de Produtos por MÃªs',
    fontsize=15,
    weight='bold'
)

axes[0].set_xlabel('')
axes[0].set_ylabel('Faturamento')

axes[0].tick_params(axis='x', rotation=45)

# Remove notaÃ§Ã£o cientÃ­fica
axes[0].yaxis.set_major_formatter(FuncFormatter(currency_formatter))

# Remove bordas desnecessÃ¡rias
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

# =========================
# Gráfico 2 - Frete
# =========================
sns.lineplot(
    data=monthly_revenue,
    x='purchase_month',
    y='freight_value',
    marker='o',
    linewidth=2.5,
    markersize=8,
    color='#f97316',
    ax=axes[1]
)

axes[1].set_title(
    'Valor de Frete por MÃªs',
    fontsize=15,
    weight='bold'
)

axes[1].set_xlabel('')
axes[1].set_ylabel('Valor de Frete')

axes[1].tick_params(axis='x', rotation=45)

axes[1].yaxis.set_major_formatter(FuncFormatter(currency_formatter))

axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

plt.tight_layout()

save_current_figure(FIGURES_DIR / '01_valores_por_mes.png')

plt.show()

top_categories = (
    delivered.groupby('dominant_category')
    .agg(orders=('order_id', 'nunique'), products_value=('products_value', 'sum'), avg_review=('review_score', 'mean'))
    .query('orders >= 100')
    .sort_values('products_value', ascending=False)
    .head(15)
    .reset_index()
)
display(top_categories)


##### Observações
- O faturamente apresentou crescimento consistente ao longo de 2017, acompanhado por aumento proporcional nos custos logí­sticos

## Reviews e experiência de entrega

Avalio a distribuição de notas e a relação inicial entre atraso e avaliação. A leitura causal fica para depois; aqui estou buscando sinais exploratórios.

In [ ]:
sns.set_theme(style='whitegrid')

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

# ==========================================
# Distribuiçãoo das avaliações
# ==========================================


sns.histplot(
    data=delivered,
    x=delivered['review_score'].astype(int),
    discrete=True,
    stat='percent',
    ax=axes[0],
    color='#4C78A8'
)

axes[0].set_title(
    'DistribuiÃ§Ã£o das AvaliaÃ§Ãµes',
    fontsize=15,
    weight='bold'
)

axes[0].set_xlabel('Nota da avaliaÃ§Ã£o')
axes[0].set_ylabel('Percentual de pedidos')

# Formata eixo Y como %
axes[0].yaxis.set_major_formatter(PercentFormatter())

# Remove bordas
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

# Labels formatados
for container in axes[0].containers:
    labels = [
        f'{bar.get_height():.1f}%'
        for bar in container
    ]

    axes[0].bar_label(
        container,
        labels=labels,
        padding=3,
        fontsize=15
    )

# ==========================================
# Distribuição de atraso
# ==========================================
sns.histplot(
    data=delivered,
    x='delay_days',
    bins=60,
    ax=axes[1],
    color='#F58518',
    edgecolor='white',
    alpha=0.9
)

axes[1].axvline(
    0,
    color='black',
    linestyle='--',
    linewidth=1.5,
    label='Entrega no prazo'
)

axes[1].set_title(
    'DistribuiÃ§Ã£o do Atraso nas Entregas',
    fontsize=15,
    weight='bold'
)

axes[1].set_xlabel('Dias de atraso')
axes[1].set_ylabel('Quantidade de pedidos')

axes[1].set_xlim(-40, 40)

axes[1].legend(frameon=False)

axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

plt.tight_layout()

save_current_figure(FIGURES_DIR / '01_reviews_e_atraso.png')

plt.show()

late_review_summary = (
    delivered.dropna(subset = ['is_late', 'is_low_review'])
    .groupby('is_late')
    .agg(orders = ('order_id', 'nunique'), avg_review = ('review_score', 'mean'), low_review_rate = ('is_low_review', 'mean'))
    .reset_index()
)

late_review_summary['low_review_rate'] = late_review_summary['low_review_rate']*100
display(late_review_summary)

##### Observações

- É possível notar que existe uma distribuição fortemente enviesada para a nota 5, enquanto a classe de avaliações ruins (`<= 2`) é minoritária.  
Portanto: ao desenvolver um modelo de ranqueamento/classificação, esse desbalanceamento deverá ser cuidadosamente considerado. Métricas como precision e recall serão priorizadas em vez de apenas acurácia.

- A satisfação média dos clientes é elevada, mas isso torna os casos de insatisfação ainda mais relevantes, pois representam exceções associadas a falhas importantes na experiência do cliente.

- Observando o gráfico de distribuição de dias de atraso, percebe-se que muitos pedidos foram entregues antes do prazo (`delay_days < 0`). Ainda assim, existe uma cauda longa de atrasos positivos.  
Portanto: a operação logística parece funcionar bem na maior parte do tempo, mas quando ocorrem falhas, elas tendem a gerar atrasos significativamente elevados.

### Conclusão

Os atrasos aparecem como o principal fator associado à insatisfação do cliente, aumentando em mais de cinco vezes a probabilidade de avaliações negativas. Embora a análise observacional não permita afirmar causalidade direta, o efeito identificado é forte, consistente e estatisticamente relevante, justificando priorização operacional na redução de atrasos logísticos.

### Nota metodológica

Os resultados desta etapa já fornecem forte evidência favorável à hipótese H1. Ainda assim, análises complementares serão realizadas nas próximas etapas para reforçar, refinar ou eventualmente contestar essa hipótese sob diferentes perspectivas.

## Armazena registros

In [ ]:
output_parquet = INTERIM_DATA_DIR / 'order_level_dataset.parquet'
output_csv = INTERIM_DATA_DIR / 'order_level_dataset.csv'

try:
    order_level.to_parquet(output_parquet, index=False)
    print(f'Base integrada salva em: {output_parquet}')
except Exception as exc:
    print(f'Parquet indisponivel ({exc}). Salvando CSV.')
    order_level.to_csv(output_csv, index=False)
    print(f'Base integrada salva em: {output_csv}')